# Sociodemographic, Substance Use, Doping Knowledge, Competitive Achievement, and Training Measures from Adult Swedish Budo & Combat Sport Federation Athletes Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (FAIR^2 dataset).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.nb1s-m2wt/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Preview basic metadata
metadata = dataset.metadata
print("Name:", metadata.name)
print("Description:", metadata.description)
print("Published:", metadata.datePublished)
print("License:", metadata.license)
print("Keywords:", metadata.keywords)
print("RecordSets:", [rs['@id'] for rs in getattr(metadata, 'recordSet', [])])

## 2. Data Overview
Review available record sets (by `@id`), their fields, and columns. Note: All references to dataset entities (record sets, fields, columns) use their `@id` fields for consistency.

Let's list all record sets, their fields (`cr:field`), and columns (`cr:column`) with their `@id`, using the metadata object.

In [ ]:
# List all available record sets and their fields/columns by @id
record_sets = getattr(metadata, 'recordSet', [])
all_record_set_ids = []
record_set_fields_columns = {}

for rs in record_sets:
    rs_id = rs['@id']
    all_record_set_ids.append(rs_id)
    fields = [f['@id'] for f in rs.get('field', [])]
    columns = [c['@id'] for c in rs.get('column', [])]
    record_set_fields_columns[rs_id] = {
        'fields': fields,
        'columns': columns
    }
    print(f"RecordSet @id: {rs_id}")
    print(f"  Fields (@id): {fields}")
    print(f"  Columns (@id): {columns}\n")

# As an example, preview the first few records (with their field @id)
if all_record_set_ids:
    example_record_set_id = all_record_set_ids[0]
    print(f"Preview sample records from RecordSet: {example_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(record)
        if i >= 2:  # print only first 3
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll demonstrate for all available record sets.

In [ ]:
# Extract data from each record set, using their @id
dataframes = {}

for rs_id in all_record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"RecordSet: {rs_id} -> columns (@id): {df.columns.tolist()}")
    print(df.head(2))

# Select the first record set for further analysis
primary_record_set_id = all_record_set_ids[0] if all_record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, categorizing data, etc.

We demonstrate these operations using the primary record set (referenced by its `@id`).

In [ ]:
# EDA on selected numeric and categorical fields by their @id
rs_id = primary_record_set_id
df = dataframes.get(rs_id) if rs_id else None

if df is not None:
    # Determine relevant field @id
    field_ids = record_set_fields_columns[rs_id]['fields']
    # Try to select a numeric field
    numeric_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    else:
        # Fallback: try to select by known column @id
        numeric_field_id = df.columns[0]

    # Set a threshold and filter
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a categorical field
    group_candidates = [col for col in df.columns if df[col].dtype == 'object']
    if group_candidates:
        group_field_id = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields using the loaded DataFrame (`primary_record_set_id`, using `@id`).

We'll demonstrate a histogram for the numeric field and a boxplot grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization using @id fields
if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_candidates:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"'{numeric_field_id}' grouped by '{group_field_id}'")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded metadata and records using Croissant schema URL.
- All dataset references use `@id` fields for each entity.
- Demonstrated overview, extraction, EDA, and visualization.
- The dataset contains diverse sociodemographic, substance use, doping and training measures from Swedish Budo & Combat Sport Federation athletes.
- Further analysis can be conducted using field and column `@id`s following Croissant standards for interoperability.